In [82]:
import duckdb
con = duckdb.connect("hmda.db")


In [83]:
con.sql("DESCRIBE SELECT * FROM 'data/state_NC.csv'")

┌───────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│            column_name            │ column_type │  null   │   key   │ default │  extra  │
│              varchar              │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ activity_year                     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ lei                               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ derived_msa-md                    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ state_code                        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ county_code                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ census_tract                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ conforming_loan_limit             │ VARCHAR     │ YES     │ NULL    │ NULL    

In [84]:
import pandas as pd
pd.set_option('display.max_rows', None)
con.sql("DESCRIBE SELECT * FROM read_csv('data/state_NC.csv', ignore_errors=true)").df()

,column_name,column_type,null,key,default,extra
0,activity_year,BIGINT,YES,None,None,None
1,lei,VARCHAR,YES,None,None,None
2,derived_msa-md,BIGINT,YES,None,None,None
3,state_code,VARCHAR,YES,None,None,None
4,county_code,VARCHAR,YES,None,None,None
5,census_tract,VARCHAR,YES,None,None,None
6,conforming_loan_limit,VARCHAR,YES,None,None,None
7,derived_loan_product_type,VARCHAR,YES,None,None,None
8,derived_dwelling_category,VARCHAR,YES,None,None,None
9,derived_ethnicity,VARCHAR,YES,None,None,None


In [85]:
con.sql("""CREATE OR REPLACE TABLE raw AS
SELECT * FROM read_csv('data/state_NC.csv', ignore_errors=true, store_rejects=true)""")
con.sql("SELECT * FROM reject_errors")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────┬────────┬────────────────────┬───────────────┬────────────┬───────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────┬──────────────────────────────────────────┐
│ scan_id │ file_id │  line  │ line_byte_position │ byte_position │ column_idx │            column_name            │                                                              error_type                                                              │                  csv_line                   │              error_message               │
│ uint64  │ uint64  │ uint64 │       uint64       │    uint64     │   uint64   │              varchar              │ enum('cast', 'missing columns', 'too many columns', 'unquoted value', 'line size over maximum', 'invalid encoding', 'invalid state') │                   varchar                   │                 varc

186 Rows failed on CSV parsing and were excluded at load.

In [86]:
con.sql("SELECT * FROM raw LIMIT 5")

┌───────────────┬──────────────────────┬────────────────┬────────────┬─────────────┬──────────────┬───────────────────────┬───────────────────────────┬──────────────────────────────────────┬─────────────────────────┬────────────────────┬───────────────────┬──────────────┬────────────────┬─────────────┬───────────┬──────────────┬─────────────┬──────────────────┬─────────────────────────┬────────────────────────────────┬─────────────┬─────────────────────┬───────────────┬─────────────┬──────────────┬──────────────────┬───────────────────────┬─────────────────────┬─────────────────┬────────────────┬───────────┬─────────────────────────┬───────────────────┬───────────────────────┬───────────────────────┬─────────────────┬──────────────────────────────┬────────────────┬─────────────────────┬────────────────┬─────────────────────────────────────────┬──────────────────────────────────────────┬─────────────┬──────────────────────────────┬─────────┬──────────────────────┬───────────────────────

In [87]:
con.sql("SELECT action_taken, COUNT(*) FROM raw GROUP BY 1 ORDER BY 2 DESC")

┌──────────────┬──────────────┐
│ action_taken │ count_star() │
│    int64     │    int64     │
├──────────────┼──────────────┤
│            1 │       259717 │
│            3 │        81317 │
│            4 │        62586 │
│            6 │        58717 │
│            5 │        32513 │
│            2 │        15405 │
│            8 │         2565 │
│            7 │          395 │
└──────────────┴──────────────┘

In [88]:
cols = con.execute("DESCRIBE raw").df()["column_name"].tolist()
results = []
for col in cols:
    q = f"""
    SELECT '{col}' AS column_name,
    COUNT(*) - COUNT("{col}") AS null_count,
    COUNT(DISTINCT "{col}") AS distinct_values,
    FROM raw
"""
    results.append(con.execute(q).df())
profile = pd.concat(results, ignore_index=True)
print(profile)


                                 column_name  null_count  distinct_values
0                              activity_year           0                1
1                                        lei           0             1200
2                             derived_msa-md           0               17
3                                 state_code           0                1
4                                county_code           0              101
5                               census_tract           0             2631
6                      conforming_loan_limit           0                3
7                  derived_loan_product_type           0                8
8                  derived_dwelling_category           0                4
9                          derived_ethnicity           0                5
10                              derived_race           0                9
11                               derived_sex           0                4
12                              action

In [89]:
con.sql("""
CREATE OR REPLACE TABLE loans AS
SELECT county_code, derived_ethnicity, derived_race, derived_sex, action_taken, purchaser_type, loan_type, TRY_CAST(loan_amount AS DOUBLE) AS loan_amount, TRY_CAST(interest_rate AS DOUBLE) AS interest_rate, TRY_CAST(loan_to_value_ratio AS DOUBLE) AS loan_to_value_ratio,
TRY_CAST(income AS DOUBLE) AS income, applicant_credit_score_type, "applicant_ethnicity-1" AS applicant_ethnicity1, "applicant_ethnicity-2" AS applicant_ethnicity2, applicant_ethnicity_observed, "applicant_race-1" AS applicant_race1, applicant_race_observed,
applicant_sex, applicant_sex_observed, lei,   applicant_age as age, debt_to_income_ratio, loan_purpose, lien_status, "denial_reason-1" AS denial_reason1, "denial_reason-2" AS denial_reason2, "denial_reason-3" AS denial_reason3, "denial_reason-4" as denial_reason4,
rate_spread, TRY_CAST(property_value AS DOUBLE) AS property_value, occupancy_type, business_or_commercial_purpose, reverse_mortgage, "open-end_line_of_credit" as open_end_line_of_credit, preapproval, total_units
FROM raw
""") 

con.sql("SELECT * FROM loans LIMIT 5")


┌─────────────┬─────────────────────────┬────────────────────┬───────────────────┬──────────────┬────────────────┬───────────┬─────────────┬───────────────┬─────────────────────┬────────┬─────────────────────────────┬──────────────────────┬──────────────────────┬──────────────────────────────┬─────────────────┬─────────────────────────┬───────────────┬────────────────────────┬──────────────────────┬─────────┬──────────────────────┬──────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬────────────────────────────────┬──────────────────┬─────────────────────────┬─────────────┬─────────────┐
│ county_code │    derived_ethnicity    │    derived_race    │    derived_sex    │ action_taken │ purchaser_type │ loan_type │ loan_amount │ interest_rate │ loan_to_value_ratio │ income │ applicant_credit_score_type │ applicant_ethnicity1 │ applicant_ethnicity2 │ applicant_ethnicity_observed │ applicant_race1 │ 

In [90]:
con.sql("SELECT COUNT(*), action_taken FROM loans GROUP BY action_taken")

┌──────────────┬──────────────┐
│ count_star() │ action_taken │
│    int64     │    int64     │
├──────────────┼──────────────┤
│       259717 │            1 │
│        15405 │            2 │
│        81317 │            3 │
│        62586 │            4 │
│        32513 │            5 │
│        58717 │            6 │
│          395 │            7 │
│         2565 │            8 │
└──────────────┴──────────────┘

In [91]:
con.sql("""SELECT 'occupancy_type' AS col, occupancy_type::VARCHAR AS value, COUNT(*) AS n
FROM loans GROUP BY 2
UNION ALL
SELECT 'business_or_commercial_purpose', business_or_commercial_purpose::VARCHAR, COUNT(*)
FROM loans GROUP BY 2
UNION ALL
SELECT 'reverse_mortgage', reverse_mortgage::VARCHAR, COUNT(*)
FROM loans GROUP BY 2
UNION ALL
SELECT 'open_end_line_of_credit', open_end_line_of_credit::VARCHAR, COUNT(*)
FROM loans GROUP BY 2
UNION ALL
SELECT 'total_units', total_units::VARCHAR, COUNT(*)
FROM loans GROUP BY 2
ORDER BY col, n DESC""")

┌────────────────────────────────┬─────────┬────────┐
│              col               │  value  │   n    │
│            varchar             │ varchar │ int64  │
├────────────────────────────────┼─────────┼────────┤
│ business_or_commercial_purpose │ 2       │ 480789 │
│ business_or_commercial_purpose │ 1       │  29797 │
│ business_or_commercial_purpose │ 1111    │   2629 │
│ occupancy_type                 │ 1       │ 459943 │
│ occupancy_type                 │ 3       │  40269 │
│ occupancy_type                 │ 2       │  13003 │
│ open_end_line_of_credit        │ 2       │ 403465 │
│ open_end_line_of_credit        │ 1       │ 107071 │
│ open_end_line_of_credit        │ 1111    │   2679 │
│ reverse_mortgage               │ 2       │ 508660 │
│ reverse_mortgage               │ 1111    │   2679 │
│ reverse_mortgage               │ 1       │   1876 │
│ total_units                    │ 1       │ 508629 │
│ total_units                    │ 2       │   2568 │
│ total_units               

In [115]:
print(con.sql("""
SELECT
    all_rows,
    all_rows - principal_res              AS dropped_occupancy,
    principal_res - plus_consumer         AS dropped_business,
    plus_consumer - plus_reverse_mortgage AS dropped_reverse,
    plus_reverse_mortgage - plus_oelec    AS dropped_open_end,
    plus_oelec - plus_total_units         AS dropped_units,
    plus_total_units - plus_action_taken AS dropped_actions,
    plus_action_taken                    AS final_universe
FROM (
    SELECT
      COUNT(*) AS all_rows,
      COUNT(*) FILTER (WHERE occupancy_type = 1) AS principal_res,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2) AS plus_consumer,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2 AND reverse_mortgage = 2) AS plus_reverse_mortgage,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2 AND reverse_mortgage = 2 AND open_end_line_of_credit = 2) AS plus_oelec,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2 AND reverse_mortgage = 2 AND open_end_line_of_credit = 2 AND total_units IN ('1','2','3','4')) AS plus_total_units,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2 AND reverse_mortgage = 2 AND open_end_line_of_credit = 2 AND total_units IN ('1','2','3','4') AND action_taken in ('1', '2', '3')) AS plus_action_taken
    FROM loans
)
""").df().to_markdown(index=False))

|   all_rows |   dropped_occupancy |   dropped_business |   dropped_reverse |   dropped_open_end |   dropped_units |   dropped_actions |   final_universe |
|-----------:|--------------------:|-------------------:|------------------:|-------------------:|----------------:|------------------:|-----------------:|
|     513215 |               53272 |               2520 |              1912 |              99190 |               3 |            125291 |           231027 |


In [116]:
print(con.sql("""
    SELECT
      COUNT(*) AS all_rows,
      COUNT(*) FILTER (WHERE occupancy_type = 1) AS principal_res,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2) AS plus_consumer,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2 AND reverse_mortgage = 2) AS plus_reverse_mortgage,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2 AND reverse_mortgage = 2 AND open_end_line_of_credit = 2) AS plus_oelec,
      COUNT(*) FILTER (WHERE occupancy_type = 1 AND business_or_commercial_purpose = 2 AND reverse_mortgage = 2 AND open_end_line_of_credit = 2 AND total_units IN ('1','2','3','4')) AS plus_total_units
    FROM loans
""").df().to_markdown(index=False))

|   all_rows |   principal_res |   plus_consumer |   plus_reverse_mortgage |   plus_oelec |   plus_total_units |
|-----------:|----------------:|----------------:|------------------------:|-------------:|-------------------:|
|     513215 |          459943 |          457423 |                  455511 |       356321 |             356318 |


|   all_rows |   dropped_occupancy |   dropped_business |   dropped_reverse |   dropped_open_end |   dropped_units |   dropped_actions |   final_universe |
|-----------:|--------------------:|-------------------:|------------------:|-------------------:|----------------:|------------------:|-----------------:|
|     513215 |               53272 |               2520 |              1912 |              99190 |               3 |            125291 |           231027 

2679 rows excluded from partially-exempt lenders where product flags unreported, <1% of applications. Remaining exclusions were investment/second homes, reverse mortgages,open line of credit, or 5+ unit properties. The final dataset after all exclusions is 231027 rows.

In [122]:
con.sql("""CREATE OR REPLACE VIEW loans_analysis AS
SELECT * FROM loans
WHERE occupancy_type = 1
  AND business_or_commercial_purpose = 2
  AND reverse_mortgage = 2
  AND open_end_line_of_credit = 2
  AND total_units IN ('1','2','3','4')
  AND action_taken in ('1','2','3')""")

## Denial rates by Race

Denominator only includes codes 1/2/3. Others excluded are preapproval, withdrawn, purchased loans, incomplete files. 

In [123]:
con.sql("""SELECT COUNT(*) FILTER (WHERE action_taken = 3) / COUNT(*) AS denial_rate, COUNT(*) AS total, derived_race
FROM loans_analysis
GROUP BY derived_race 
ORDER BY denial_rate""")

┌─────────────────────┬────────┬───────────────────────────────────────────┐
│     denial_rate     │ total  │               derived_race                │
│       double        │ int64  │                  varchar                  │
├─────────────────────┼────────┼───────────────────────────────────────────┤
│ 0.11793080505655355 │  12024 │ Asian                                     │
│ 0.15433251596715328 │ 140288 │ White                                     │
│ 0.15719130261320566 │   5013 │ Joint                                     │
│ 0.24969051079175414 │  37158 │ Race Not Available                        │
│  0.2571428571428571 │     35 │ Free Form Text Only                       │
│ 0.26785714285714285 │    392 │ Native Hawaiian or Other Pacific Islander │
│ 0.30543367193604615 │  33274 │ Black or African American                 │
│  0.3116219667943806 │    783 │ 2 or more minority races                  │
│  0.3524271844660194 │   2060 │ American Indian or Alaska Native          │

Black applicants are denied at a 30.5% vs 15.4% rate compared to White applicants, roughly a 15 point difference. The highest denial rate is in American Indian and Alaskan Natives (35.2%, n = 2,060). These are consistent with national HMDA patterns. 

This is not evidence of discrimination just yet, no controls for DTI, income, LTV, and there is not a credit score in this data. 

## Denial Rate By Age, Income, Sex

Same denominator as race. These are probably less significant, but checking anyways just to get a baseline.

In [124]:
con.sql("""SELECT COUNT(*) FILTER (WHERE action_taken = 3) / COUNT(*) AS denial_rate, COUNT(*) AS total, age
FROM loans_analysis
GROUP BY age
ORDER BY denial_rate""")

┌─────────────────────┬───────┬─────────┐
│     denial_rate     │ total │   age   │
│       double        │ int64 │ varchar │
├─────────────────────┼───────┼─────────┤
│ 0.14201344227250493 │ 55199 │ 25-34   │
│ 0.16811955168119552 │ 11242 │ <25     │
│ 0.17786583118319205 │ 54260 │ 35-44   │
│  0.2126549249836921 │ 42924 │ 45-54   │
│ 0.22397873531057638 │ 35740 │ 55-64   │
│   0.234049245147376 │ 22256 │ 65-74   │
│ 0.26199626660810366 │  9107 │ >74     │
│  0.9163879598662207 │   299 │ 8888    │
└─────────────────────┴───────┴─────────┘

Older applicants are denied at higher rates, but it is a very gradual increase. 299 applicants did not provide age.

In [125]:
con.sql("""SELECT COUNT(*) FILTER (WHERE action_taken = 3) / COUNT(*) AS denial_rate, COUNT(*) AS total, derived_sex
FROM loans_analysis
GROUP BY derived_sex
ORDER BY denial_rate""")

┌─────────────────────┬───────┬───────────────────┐
│     denial_rate     │ total │    derived_sex    │
│       double        │ int64 │      varchar      │
├─────────────────────┼───────┼───────────────────┤
│   0.144147559413318 │ 76498 │ Joint             │
│  0.1896170972763585 │ 82867 │ Male              │
│ 0.23341461211957013 │ 55365 │ Female            │
│ 0.28956249616493834 │ 16297 │ Sex Not Available │
└─────────────────────┴───────┴───────────────────┘

There was a 4.4% difference in denial rates for female applicants compared to male applicants. For over 16,000 the sex was not available, and another 76,498 were joint applications. 

In [126]:
con.sql("""SELECT
    CASE
        WHEN income IS NULL THEN 'e: not reported'
        WHEN income < 50 THEN 'a: <50k'
        WHEN income < 100 THEN 'b: 50-100k'
        WHEN income < 150 THEN 'c: 100-150k'
        ELSE 'd: 150k+'
    END AS income_band,
    COUNT(*) AS n,
    AVG(CASE WHEN action_taken = 3 THEN 1.0 ELSE 0 END) AS denial_rate
FROM loans_analysis
GROUP BY income_band
ORDER BY income_band""")

┌─────────────────┬───────┬─────────────────────┐
│   income_band   │   n   │     denial_rate     │
│     varchar     │ int64 │       double        │
├─────────────────┼───────┼─────────────────────┤
│ a: <50k         │ 27900 │  0.4535483870967742 │
│ b: 50-100k      │ 83869 │ 0.20675100454279888 │
│ c: 100-150k     │ 49633 │ 0.14004795196744102 │
│ d: 150k+        │ 52226 │ 0.09627388656990771 │
│ e: not reported │ 17399 │  0.1384562331168458 │
└─────────────────┴───────┴─────────────────────┘

As expected, as income increases, denial rate decreases. It is a much steeper drop off at the lower incomes. 